# Manuscript Figures

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import sys
from pathlib import Path
import json

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import PROJECT_ROOT, DATA_ROOT, PLOTS_ROOT, prepare_directory
from src.utilities import (
    read_datasets_from_label,
    read_denoise_results,
    read_coord_transform_results,
    read_inference_results,
    get_keys_h5,
    get_attrs_h5,
)
from src.visualization import (
    plot_lines,
    format_axis,
    plot_spheres,
)
from src.modeling import (
    create_spring_pendulum_equation,
    read_inference_params,
    compute_derived_params,
)

In [ ]:
# Directory and files setup

PLOT_DIR = PLOTS_ROOT
prepare_directory(PLOT_DIR, clear=False, verbose=False)

print('PROJECT_ROOT:'.ljust(20), PROJECT_ROOT)
print('PLOT_DIR:'.ljust(20), PLOT_DIR.relative_to(PROJECT_ROOT))

In [ ]:
# Figure settings

CLOSE_FIGS = False

plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.transparent'] = True

In [ ]:
# Helpers

def shared_range_limits(*arrays, margin=0.05):
    """Return per-array limits with a shared span and array-specific centers."""
    arrays = [np.asarray(a) for a in arrays]
    centers = [0.5 * (np.nanmin(a) + np.nanmax(a)) for a in arrays]
    span = max(np.nanmax(a) - np.nanmin(a) for a in arrays)
    span = span * (1 + 2 * margin)

    return [(c - span / 2, c + span / 2) for c in centers]

def get_interval_spring_params(inference_results, interval_idx, derived_param_names=('L0',)):
    """Return fitted spring-pendulum parameters for one inference interval."""
    param_names = inference_results['spring_pendulum_params']['param_names']
    params_fitted = inference_results['spring_pendulum_params']['params_fitted'][:, interval_idx]
    spring_pendulum_params = dict(zip(param_names, np.nan_to_num(params_fitted, nan=0.0)))

    derived_params, _ = compute_derived_params(
        spring_pendulum_params,
        derived_param_names=list(derived_param_names),
    )
    spring_pendulum_params.update(derived_params)

    return spring_pendulum_params

def evaluate_fitted_surface(
    feature_df, data_mask, inference_results, interval_idx,
    x_key, y_key, x_limits, y_limits, mesh_size, breakpoint_key,
):
    """Evaluate the fitted vertical-acceleration surface for one interval."""
    r_interval = np.asarray(feature_df['r'])[data_mask]
    cos_theta_interval = np.asarray(feature_df['cos_theta'])[data_mask]
    r_dot_interval = np.asarray(feature_df['r_dot_cos_theta'])[data_mask] / cos_theta_interval
    z_dot_interval = np.asarray(feature_df['z_dot'])[data_mask]

    spring_pendulum_params = get_interval_spring_params(
        inference_results,
        interval_idx,
        derived_param_names=('L0',),
    )
    spring_pendulum_fn = create_spring_pendulum_equation(
        spring_pendulum_params,
        breakpoint_key=breakpoint_key,
    )

    x_mesh, y_mesh = np.meshgrid(
        np.linspace(x_limits[0], x_limits[1], mesh_size),
        np.linspace(y_limits[0], y_limits[1], mesh_size),
    )

    surface_features = {
        'r': np.full_like(x_mesh, np.nanmean(r_interval)),
        'cos_theta': np.full_like(x_mesh, np.nanmean(cos_theta_interval)),
        'r_dot': np.full_like(x_mesh, np.nanmean(r_dot_interval)),
    }
    surface_features[x_key] = x_mesh
    surface_features[y_key] = y_mesh

    z_mesh = spring_pendulum_fn(
        surface_features['r'],
        surface_features['cos_theta'],
        surface_features['r_dot'],
        np.nanmean(z_dot_interval),
    )

    breakpoint_val = spring_pendulum_params[breakpoint_key]
    surface_masks = {
        'slack': surface_features['r'] <= breakpoint_val,
        'taut': surface_features['r'] > breakpoint_val,
    }

    return x_mesh, y_mesh, z_mesh, surface_masks

## Denoising parameters

In [ ]:
# Print denoising parameters

DENOISE_TYPES = ['tangle_00', 'tangle_01']
DENOISE_STAGE = '02_denoise'
DENOISE_COORD_KEYS = ['x', 'y', 'z']

DENOISE_PARAM = 'smoother__bandwidth'
DENOISE_FLOAT_FMT = '.3g'
DENOISE_LJUST = 12

columns = [
    ('type', DENOISE_LJUST),
    ('label', DENOISE_LJUST),
    ('h_x', DENOISE_LJUST),
    ('mse_x', DENOISE_LJUST),
    ('h_y', DENOISE_LJUST),
    ('mse_y', DENOISE_LJUST),
    ('h_z', DENOISE_LJUST),
    ('mse_z', DENOISE_LJUST),
]

header = ''.join(name.ljust(width) for name, width in columns)
print(header)
print('-' * len(header))

for tangle_type in DENOISE_TYPES:
    denoise_path = DATA_ROOT / tangle_type / f'{DENOISE_STAGE}.h5'

    for data_label in get_keys_h5(denoise_path):
        _, _, denoise_config, coordinate_cv_results = read_denoise_results(
            denoise_path,
            data_label,
        )

        row = {'type': tangle_type,'label': data_label}

        for coord_key in DENOISE_COORD_KEYS:
            cv_results = coordinate_cv_results[coord_key]
            row[f'h_{coord_key}'] = cv_results['best_params'][DENOISE_PARAM]
            row[f'mse_{coord_key}'] = np.nanmin(cv_results['mean_test_score'])

        print(
            f"{row['type']:<{DENOISE_LJUST}}"
            f"{row['label']:<{DENOISE_LJUST}}"
            f"{row['h_x']:<{DENOISE_LJUST}{DENOISE_FLOAT_FMT}}"
            f"{row['mse_x']:<{DENOISE_LJUST}{DENOISE_FLOAT_FMT}}"
            f"{row['h_y']:<{DENOISE_LJUST}{DENOISE_FLOAT_FMT}}"
            f"{row['mse_y']:<{DENOISE_LJUST}{DENOISE_FLOAT_FMT}}"
            f"{row['h_z']:<{DENOISE_LJUST}{DENOISE_FLOAT_FMT}}"
            f"{row['mse_z']:<{DENOISE_LJUST}{DENOISE_FLOAT_FMT}}"
        )

## Fit results

### Time-resolved parameter inference

In [ ]:
# Inference parameters and metrics configurations

# Inference parameters

PARAM_KEYS = [
    'k_m',
    'L0',
]
PARAM_LABELS = {
    'k_m': r'$\kappa_{\mathrm{dyn}}$ (s$^{-2}$)',
    'L0': r'$\ell_{\mathrm{dyn}}$ (m)',
    'c_m': r'$c/m$ (s$^{-1}$)',
}
PARAM_COLORS = {
    'k_m': (cm.Blues(0.8), cm.Blues(0.6)),
    'L0': (cm.Oranges(0.75), cm.Oranges(0.6)),
    'c_m': (cm.Purples(0.8), cm.Purples(0.6)),
}
PARAM_LIMITS = {
    'k_m': (0, 6500),
    'L0': (0.0, 0.07),
    'c_m': (-10, 50)
}
DERIVED_PARAM_KEYS = ['L0']

# Fit metrics

METRIC_KEYS = [
    'r2_scores'
]
METRIC_LABELS = {
    'mse_scores': 'MSE',
    'r2_scores': '$R^2$',
}
METRIC_COLORS = {
    'mse_scores': 'k',
    'r2_scores': 'k',
}
METRIC_LIMITS = {
    'mse_scores': None,
    'r2_scores': (-.1, 1.1),
}
METRIC_REFVAL = {
    'mse_scores': 0,
    'r2_scores': 1,
}

In [ ]:
# Select datasets and configure visualization

# Dataset and paths

FIT_STAGE = '04_inference'
FIT_DATASETS = [
    ('tangle_00', 'tw10_low'),
    ('tangle_00', 'tw10_opt'),
    ('tangle_00', 'tw10_high')
]
FIT_DATA_PATHS = {
    dataset_key: DATA_ROOT / dataset_key[0] / f'{FIT_STAGE}.h5'
    for dataset_key in FIT_DATASETS
}
FIT_FIG_PATHS = {
    dataset_key: PLOT_DIR / f'fit__{dataset_key[0]}__{dataset_key[1]}.png'
    for dataset_key in FIT_DATASETS
}

# Visualization config

FIT_TIME_ZERO = True

FIT_TITLES = {
    'low': r'Low',
    'opt': r'Moderate',
    'high': r'High'
}
FIT_XLABEL = r'Time (s)'

FIT_AXIS_CONFIG = {
    'figsize': (3.5, 1.5 * (len(PARAM_KEYS) + len(METRIC_KEYS)) + 0.4),
    'grid': False,
    'scilimits': (-1, 1),
    'sci_axis': 'y',
    'margins': (0.0, 0.1),
}

In [ ]:
# Plot inference parameters and metrics

for dataset_key in FIT_DATASETS:

    # Read inference results
    tangle_type, data_label = dataset_key
    data_path = FIT_DATA_PATHS[dataset_key]
    fig_path = FIT_FIG_PATHS[dataset_key]

    _, _, _, _, equation, time_intervals, _ = read_inference_results(data_path, data_label)

    (time_midpoints, param_names, param_labels, 
     params_fitted, bootstrap_means, bootstrap_stds, 
     bootstrap_ci_lowers, bootstrap_ci_uppers, 
     metric_values) = read_inference_params(
        data_path, data_label,
        params_plot=PARAM_KEYS,
        derived_params=DERIVED_PARAM_KEYS,
        metrics_plot=METRIC_KEYS,
    )

    # Prepare visualization
    if FIT_TIME_ZERO:
        time_midpoints = time_midpoints - time_midpoints[0]

    param_labels = {**param_labels, **PARAM_LABELS}
    title = FIT_TITLES.get(data_label.split('_')[-1], data_label)
    
    n_param_axes = len(PARAM_KEYS)
    n_metric_axes = len(METRIC_KEYS)
    n_axes = n_param_axes + n_metric_axes

    fig, axes = plt.subplots(n_axes, 1, sharex=True)
    axes = np.atleast_1d(axes)

    # Plot fitted parameters
    for i, param_name in enumerate(param_names):
        ax = axes[i]
        point_color, range_color = PARAM_COLORS[param_name]

        ax.fill_between(
            time_midpoints,
            bootstrap_ci_lowers[i],
            bootstrap_ci_uppers[i],
            color=range_color,
            alpha=0.3,
            linewidth=0.8,
            zorder=0,
            label='95% CI',
        )

        ax.errorbar(
            time_midpoints,
            params_fitted[i],
            yerr=bootstrap_stds[i],
            fmt='none',
            ecolor=point_color,
            elinewidth=0.8,
            capsize=2,
            zorder=1,
            label=r'$\pm$ std'
        )

        plot_lines(
            [(time_midpoints, params_fitted[i])],
            ax=ax,
            color=point_color,
            linestyle='None',
            marker='o',
            markersize=3,
            zorder=2,
        )

        format_axis(
            ax,
            **FIT_AXIS_CONFIG,
            ylabel=param_labels.get(param_name, param_name),
            ylims=PARAM_LIMITS.get(param_name),
            title=title if i == 0 else None,
            legend_kwargs={'loc': 'upper right', 'fontsize': 7, 'frameon': True}
        )

    # Plot fit metrics
    for j, metric_name in enumerate(METRIC_KEYS):
        ax = axes[n_param_axes + j]

        plot_lines(
            [(time_midpoints, metric_values[j])],
            ax=ax,
            color=METRIC_COLORS.get(metric_name, 'k'),
            linewidth=0.8,
            marker='^',
            markersize=3.5,
        )
        if METRIC_REFVAL.get(metric_name) is not None:
            ax.axhline(
                METRIC_REFVAL[metric_name],
                color='gray',
                linestyle='--',
                linewidth=0.8,
                zorder=-1,
            )

        format_axis(
            ax,
            **FIT_AXIS_CONFIG,
            xlabel=FIT_XLABEL,
            ylabel=METRIC_LABELS.get(metric_name, metric_name),
            ylims=METRIC_LIMITS.get(metric_name),
        )

    fig.align_ylabels(axes)
    fig.tight_layout()
    fig.savefig(fig_path)

    if CLOSE_FIGS:
        plt.close(fig)

    print(f'Saved: {fig_path.relative_to(PROJECT_ROOT)}')

### Fitted-surface plots

In [ ]:
# Feature/target configuration

FEATURE_LABELS = {
    'r':         r'$r$ (m)',
    'cos_theta': r'$\cos{\theta}$',
    'r_dot':     r'$\dot{r}$ (m/s)',
}

TARGET_LABELS = {
    'z_ddot':    r'$\ddot{y}$ (m/s²)',
}

In [ ]:
# Select datasets and configure visualization

# Dataset and paths

FIT_SURFACE_DATASET = ('tangle_00', 'tw10_opt')
FIT_SURFACE_FEATURES_PATH = DATA_ROOT / FIT_SURFACE_DATASET[0] / f'03_coord_transform.h5'
FIT_SURFACE_PARAMS_PATH = DATA_ROOT / FIT_SURFACE_DATASET[0] / f'04_inference.h5'
FIT_SURFACE_FIG_PATH = PLOT_DIR / f'fit_surface__{FIT_SURFACE_DATASET[0]}__{FIT_SURFACE_DATASET[1]}.png'

# Visualization config

FIT_SURFACE_INTERVAL_IDXS = np.arange(0, 15)
FIT_SURFACE_FEATURE_KEYS = ('r', 'cos_theta')

FIT_SURFACE_SCATTER_CONFIG = {
    'c': 'k',
    'alpha': 1,
    'edgecolors': 'none',
}
FIT_SURFACE_POINT_RADIUS = 0.015

FIT_SURFACE_MESH_SIZE = 1000
FIT_SURFACE_BREAKPOINT_KEY = 'L0'

FIT_SURFACE_COLORS = {
    'slack': 'tab:orange',
    'taut': 'tab:blue',
}
FIT_SURFACE_ALPHA = 0.3

FIT_SURFACE_COORD_LIMIT_CONFIG = json.loads(
    get_attrs_h5(FIT_SURFACE_FEATURES_PATH)['coord_limit_config']
)
FIT_SURFACE_AXIS_CONFIG = {
    'figsize': (3.6, 3.6),
    'view_angles': (15, 240),
}

# Configure special index to highlight breakpoint

BREAKPOINT_HLT_IDX = 7
BREAKPOINT_HLT_FIG_PATH = PLOT_DIR / (
    f'breakpoint_hlt__{FIT_SURFACE_DATASET[0]}__{FIT_SURFACE_DATASET[1]}'
    f'__interval{BREAKPOINT_HLT_IDX:02d}.png'
)

BREAKPOINT_HLT_AXIS_CONFIG = {
    **FIT_SURFACE_AXIS_CONFIG,
    'figsize': (3.2, 3.2),
    'view_angles': (20, 245),
}
BREAKPOINT_HLT_LINES_CONFIG = {
    'L0': {
        'label': r'$L_0$',
        'color': PARAM_COLORS['L0'][0],
        'linestyle': '--',
        'linewidth': 1,
        'alpha': 1,
    },
    'xi': {
        'label': r'$\xi$',
        'color': PARAM_COLORS['L0'][0],
        'linestyle': ':',
        'linewidth': 1,
        'alpha': 1,
    },
}

In [ ]:
# Set up fitted-surface visualization

# Read coordinate-transform features and inference intervals
coord_datasets, data_params, coord_transform_config, equations_data, candidates_data = read_coord_transform_results(
    FIT_SURFACE_FEATURES_PATH,
    FIT_SURFACE_DATASET[1],
)
_, _, _, _, equation, time_intervals, inference_results = read_inference_results(
    FIT_SURFACE_PARAMS_PATH,
    FIT_SURFACE_DATASET[1],
)
equation_data = equations_data[equation]
feature_df = equation_data['feature_df']
target_data = np.asarray(equation_data['target_data']).squeeze()
target_name = equation_data['target_name']

x_key, y_key = FIT_SURFACE_FEATURE_KEYS

# Prepare coordinate limits and mask
coord_limit_params = FIT_SURFACE_COORD_LIMIT_CONFIG[FIT_SURFACE_DATASET[1]]
feature_limits = coord_limit_params.get('feature_limits', {})
target_limits = coord_limit_params.get('target_limits', {})

coord_mask = np.ones(len(coord_datasets['t']), dtype=bool)

for feature_name in FIT_SURFACE_FEATURE_KEYS:
    if feature_name in feature_limits:
        feature_data = np.asarray(feature_df[feature_name])
        feature_min, feature_max = feature_limits[feature_name]
        coord_mask &= (feature_data >= feature_min) & (feature_data <= feature_max)

if target_name in target_limits:
    target_min, target_max = target_limits[target_name]
    coord_mask &= (target_data >= target_min) & (target_data <= target_max)

In [ ]:
# Plot fitted-surface scatter points and fitted surfaces for selected intervals

for interval_idx in FIT_SURFACE_INTERVAL_IDXS:

    if interval_idx >= len(time_intervals):
        continue

    # Prepare interval data
    t_start, t_end = time_intervals[interval_idx]
    time_mask = ((coord_datasets['t'] >= t_start) & (coord_datasets['t'] <= t_end))
    data_mask = time_mask & coord_mask

    x_data = np.asarray(feature_df[x_key])[data_mask]
    y_data = np.asarray(feature_df[y_key])[data_mask]
    z_data = target_data[data_mask]
    coordinates = np.column_stack([x_data, y_data, z_data])

    x_limits = feature_limits.get(x_key, (np.nanmin(x_data), np.nanmax(x_data)))
    y_limits = feature_limits.get(y_key, (np.nanmin(y_data), np.nanmax(y_data)))
    z_limits = target_limits.get(target_name, (np.nanmin(z_data), np.nanmax(z_data)))

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    # Plot scatter points
    plot_spheres(
        coordinates,
        radius=FIT_SURFACE_POINT_RADIUS,
        sphere_props=FIT_SURFACE_SCATTER_CONFIG,
        ax=ax,
    )
    # Plot fitted surface
    x_mesh, y_mesh, z_mesh, surface_masks = evaluate_fitted_surface(
        feature_df, data_mask, inference_results, interval_idx,
        x_key, y_key, x_limits, y_limits,
        FIT_SURFACE_MESH_SIZE, FIT_SURFACE_BREAKPOINT_KEY,
    )
    z_mask = np.ones_like(z_mesh, dtype=bool)
    if z_limits is not None:
        z_min, z_max = z_limits
        if z_min is not None:
            z_mask &= z_mesh >= z_min
        if z_max is not None:
            z_mask &= z_mesh <= z_max

    for surface_key, surface_mask in surface_masks.items():
        plot_mask = surface_mask & z_mask
        ax.plot_surface(
            np.where(plot_mask, x_mesh, np.nan),
            np.where(plot_mask, y_mesh, np.nan),
            np.where(plot_mask, z_mesh, np.nan),
            color=FIT_SURFACE_COLORS[surface_key],
            alpha=FIT_SURFACE_ALPHA,
            linewidth=0,
            antialiased=True,
        )

    # Format axis and save figure
    format_axis(
        ax,
        **FIT_SURFACE_AXIS_CONFIG,
        xlabel=FEATURE_LABELS.get(x_key, x_key),
        ylabel=FEATURE_LABELS.get(y_key, y_key),
        zlabel=TARGET_LABELS.get(target_name, target_name),
        xlims=x_limits,
        ylims=y_limits,
        zlims=z_limits,
    )
    ax.set_title(f'Time interval: {t_start:.1f}–{t_end:.1f} s', y=1, fontsize=10)
    fig.tight_layout()

    fig_path = FIT_SURFACE_FIG_PATH.with_name(
        f'{FIT_SURFACE_FIG_PATH.stem}__interval{interval_idx:02d}{FIT_SURFACE_FIG_PATH.suffix}'
    )
    fig.savefig(fig_path, bbox_inches='tight', pad_inches=0.3)

    print(f'Saved: {fig_path.relative_to(PROJECT_ROOT)}')

    # Breakpoint highlight figure for special interval
    if interval_idx == BREAKPOINT_HLT_IDX:
        spring_pendulum_params = get_interval_spring_params(
            inference_results,
            interval_idx,
            derived_param_names=('L0',),
        )
        spring_pendulum_fn = create_spring_pendulum_equation(
            spring_pendulum_params,
            breakpoint_key=FIT_SURFACE_BREAKPOINT_KEY,
        )

        cos_theta_line = np.linspace(y_limits[0], y_limits[1], 200)
        r_dot_mean = np.nanmean(np.asarray(feature_df['r_dot_cos_theta'])[data_mask] / np.asarray(feature_df['cos_theta'])[data_mask])
        z_dot_mean = np.nanmean(np.asarray(feature_df['z_dot'])[data_mask])

        for breakpoint_key, line_config in BREAKPOINT_HLT_LINES_CONFIG.items():
            r_val = spring_pendulum_params[breakpoint_key]
            r_line = np.full_like(cos_theta_line, r_val)
            z_line = spring_pendulum_fn(r_line, cos_theta_line, r_dot_mean, z_dot_mean)

            line_mask = np.isfinite(z_line)
            if z_limits is not None:
                line_mask &= (z_line >= z_limits[0]) & (z_line <= z_limits[1])

            ax.plot(
                r_line[line_mask],
                cos_theta_line[line_mask],
                z_line[line_mask],
                **line_config,
            )

        ax.set_title('')
        format_axis(
            ax, **BREAKPOINT_HLT_AXIS_CONFIG, 
            legend_kwargs={'loc': 'best', 'bbox_to_anchor': (0.35, 0.8), 'fontsize': 8, 'frameon': True,}
        )
        
        fig.tight_layout()
        fig.savefig(BREAKPOINT_HLT_FIG_PATH, bbox_inches='tight', pad_inches=0.3)
        print(f'Saved: {BREAKPOINT_HLT_FIG_PATH.relative_to(PROJECT_ROOT)}')

    plt.close(fig)

## Trajectories

### Time series

In [ ]:
# Select dataset and configure visualization

# Dataset and paths

TRAJ_TYPE = 'tangle_00'
TRAJ_STAGE = '01_preprocess'

TRAJ_DATASET = 'tw10_opt'
TRAJ_TIME_RANGE_FRAC = (0.7, 1)

TRAJ_DATA_PATH = DATA_ROOT / TRAJ_TYPE / f'{TRAJ_STAGE}.h5'
TRAJ_FIG_PATH = PLOT_DIR / f'trajectory__{TRAJ_TYPE}__{TRAJ_DATASET}.png'

# Visualization config

TRAJ_TIME_ZERO = True
TRAJ_COORD_KEYS = ['x', 'z', 'y']

TRAJ_INCLUDE_PISTON = True

TRAJ_XLABEL = r'Time (s)'
TRAJ_YLABELS = [r'$x(t)$ (m)', r'$y(t)$ (m)', r'$z(t)$ (m)']

TRAJ_LINES_CONFIG = {
    'color': 'k',
    'linewidth': 0.8,
}

TRAJ_PISTON_LINES_CONFIG = {
    'color': 'orange',
    'linewidth': 0.8,
}

TRAJ_AXIS_CONFIG = {
    'grid': False,
    'scilimits': (-1, 1),
    'sci_axis': 'y',
    'margins': (0.0, 0.1),
    'figsize': (4.2, 3.6),
}

TRAJ_RANGE_MARGIN = 0.05

In [ ]:
# Prepare selection

# Read datasets
datasets, params = read_datasets_from_label(TRAJ_DATA_PATH, TRAJ_DATASET)
t = np.asarray(datasets['t'])
coords = [np.asarray(datasets[key]) for key in TRAJ_COORD_KEYS]
piston_coords = [np.asarray(datasets[key]) for key in [f'{key}p' for key in TRAJ_COORD_KEYS]]

# Prepare time slice and plot coordinates
n_points = len(t)
frac_start, frac_stop = TRAJ_TIME_RANGE_FRAC
idx_start = int(np.floor(frac_start * n_points))
idx_stop = int(np.ceil(frac_stop * n_points))
time_slice = slice(idx_start, idx_stop)

t_plot = t[time_slice] - t[time_slice][0] if TRAJ_TIME_ZERO else t[time_slice]
coords_plot = [coord[time_slice] for coord in coords]
piston_coords_plot = [coord[time_slice] for coord in piston_coords]

# Prepare axis limits
if TRAJ_INCLUDE_PISTON:
    limit_arrays = [
        np.concatenate([coord, piston_coord])
        for coord, piston_coord in zip(coords_plot, piston_coords_plot)
    ]
else:
    limit_arrays = coords_plot

ylims = shared_range_limits(*limit_arrays, margin=TRAJ_RANGE_MARGIN)

print(f"Time range: ({t[time_slice][0]:.3f}, {t[time_slice][-1]:.3f}) s")

In [ ]:
# Plot trajectory coordinates

fig, axes = plt.subplots(len(coords_plot), 1, sharex=True)
axes = np.atleast_1d(axes)

for i, (ax, coord, ylabel, ylim) in enumerate(zip(axes, coords_plot, TRAJ_YLABELS, ylims)):
    plot_lines(
        [(t_plot, coord)],
        ax=ax,
        **TRAJ_LINES_CONFIG,
    )

    if TRAJ_INCLUDE_PISTON:
        plot_lines(
            [(t_plot, piston_coords_plot[i])],
            ax=ax,
            **TRAJ_PISTON_LINES_CONFIG,
        )
    
    axis_config = {
        **TRAJ_AXIS_CONFIG,
        'ylabel': ylabel,
        'ylims': ylim,
    }
    if i == len(axes) - 1:
        axis_config['xlabel'] = TRAJ_XLABEL

    format_axis(ax, **axis_config)

fig.align_ylabels(axes)
fig.tight_layout()
fig.savefig(TRAJ_FIG_PATH)

if CLOSE_FIGS:
    plt.close(fig)

print(f'Saved: {TRAJ_FIG_PATH.relative_to(PROJECT_ROOT)}')

### Time windows

In [ ]:
# Configure visualization

WINDOW_INTERVAL_IDX = 7
WINDOW_COLOR = 'grey'
WINDOW_ALPHA = 0.25

WINDOW_COORD_KEY = 'z'
WINDOW_INCLUDE_PISTON = True

WINDOW_AXIS_CONFIG = {
    **TRAJ_AXIS_CONFIG,
    'xlabel': r'Time (s)',
    'ylabel': r'$y(t)$ (m)',
    'figsize': (4.2, 1.6)
}

WINDOW_FIG_PATH = PLOT_DIR / f'trajectory_time_window__{TRAJ_TYPE}__{TRAJ_DATASET}__interval{WINDOW_INTERVAL_IDX:02d}.png'

In [ ]:
# Plot time window for selected coordinate

# Read inference intervals for the selected trajectory dataset
_, _, _, _, _, window_time_intervals, _ = read_inference_results(
    DATA_ROOT / TRAJ_TYPE / '04_inference.h5',
    TRAJ_DATASET,
)

window_start, window_end = window_time_intervals[WINDOW_INTERVAL_IDX]
time_origin = t[time_slice][0] if TRAJ_TIME_ZERO else 0.0
window_start_plot = window_start - time_origin
window_end_plot = window_end - time_origin

# Prepare selected coordinate
coord = np.asarray(datasets[WINDOW_COORD_KEY])
coord_plot = coord[time_slice]

if WINDOW_INCLUDE_PISTON:
    piston_coord = np.asarray(datasets[f'{WINDOW_COORD_KEY}p'])
    piston_coord_plot = piston_coord[time_slice]
    ylim = shared_range_limits(
        np.concatenate([coord_plot, piston_coord_plot]),
        margin=TRAJ_RANGE_MARGIN,
    )[0]
else:
    piston_coord_plot = None
    ylim = shared_range_limits(coord_plot, margin=TRAJ_RANGE_MARGIN)[0]

# Plot
fig, ax = plt.subplots()

ax.axvspan(
    window_start_plot,
    window_end_plot,
    color=WINDOW_COLOR,
    alpha=WINDOW_ALPHA,
    linewidth=0,
    zorder=-10,
)

plot_lines(
    [(t_plot, coord_plot)],
    ax=ax,
    **TRAJ_LINES_CONFIG,
    zorder=2,
)

if WINDOW_INCLUDE_PISTON:
    plot_lines(
        [(t_plot, piston_coord_plot)],
        ax=ax,
        **TRAJ_PISTON_LINES_CONFIG,
        zorder=2,
    )

format_axis(
    ax,
    **WINDOW_AXIS_CONFIG,
    xlims=(t_plot[0], t_plot[-1]),
    ylims=ylim,
)

fig.tight_layout()
fig.savefig(WINDOW_FIG_PATH)

if CLOSE_FIGS:
    plt.close(fig)

print(f'Saved: {WINDOW_FIG_PATH.relative_to(PROJECT_ROOT)}')

### Coordinate transform

In [ ]:
# Coordinate-transform configuration

CARTESIAN_COORD_LABELS = {
    'x': r'$x$ (m)',
    'y': r'$z$ (m)',
}

In [ ]:
# Configure visualization

# Dataset and paths

COORD_TRANSFORM_DATASET = ('tangle_00', 'tw10_low')
COORD_TRANSFORM_INTERVAL_IDXS = [8, 9]

COORD_TRANSFORM_INTERVAL_SUFFIX = (
    f'interval{COORD_TRANSFORM_INTERVAL_IDXS[0]:02d}'
    if len(COORD_TRANSFORM_INTERVAL_IDXS) == 1
    else f'intervals{COORD_TRANSFORM_INTERVAL_IDXS[0]:02d}-{COORD_TRANSFORM_INTERVAL_IDXS[-1]:02d}'
)

COORD_TRANSFORM_FEATURES_PATH = DATA_ROOT / COORD_TRANSFORM_DATASET[0] / '03_coord_transform.h5'
COORD_TRANSFORM_PARAMS_PATH = DATA_ROOT / COORD_TRANSFORM_DATASET[0] / '04_inference.h5'

COORD_TRANSFORM_XY_FIG_PATH = (
    PLOT_DIR
    / f'coord_transform_xy__{COORD_TRANSFORM_DATASET[0]}__{COORD_TRANSFORM_DATASET[1]}__{COORD_TRANSFORM_INTERVAL_SUFFIX}.png'
)
COORD_TRANSFORM_FEATURE_FIG_PATH = (
    PLOT_DIR
    / f'coord_transform_features__{COORD_TRANSFORM_DATASET[0]}__{COORD_TRANSFORM_DATASET[1]}__{COORD_TRANSFORM_INTERVAL_SUFFIX}.png'
)

# Visualization config

COORD_TRANSFORM_CARTESIAN_KEYS = ('x', 'y')
COORD_TRANSFORM_FEATURE_KEYS = ('r', 'cos_theta')
COORD_TRANSFORM_MASK_FEATURE_KEYS = ('r', 'cos_theta')

COORD_TRANSFORM_SCATTER_CONFIG = {
    'c': 'k',
    'alpha': 1,
    'edgecolors': 'none',
}
COORD_TRANSFORM_POINT_RADIUS = 0.015
COORD_TRANSFORM_RANGE_MARGIN = 0.05

COORD_TRANSFORM_COORD_LIMIT_CONFIG = json.loads(
    get_attrs_h5(COORD_TRANSFORM_FEATURES_PATH)['coord_limit_config']
)

COORD_TRANSFORM_CARTESIAN_AXIS_CONFIG = {
    'figsize': (3.6, 3.6),
    'view_angles': (25, 45),
    'grid': False,
}
COORD_TRANSFORM_FEATURE_AXIS_CONFIG = {
    'figsize': (3.6, 3.6),
    'view_angles': (15, 245),
    'grid': False,
}

In [ ]:
# Set up coordinate-transform visualization

# Read coordinate-transform features and inference intervals
ct_coord_datasets, ct_data_params, ct_coord_transform_config, ct_equations_data, ct_candidates_data = read_coord_transform_results(
    COORD_TRANSFORM_FEATURES_PATH,
    COORD_TRANSFORM_DATASET[1],
)
_, _, _, _, ct_equation, ct_time_intervals, ct_inference_results = read_inference_results(
    COORD_TRANSFORM_PARAMS_PATH,
    COORD_TRANSFORM_DATASET[1],
)

ct_equation_data = ct_equations_data[ct_equation]
ct_feature_df = ct_equation_data['feature_df']
ct_target_data = np.asarray(ct_equation_data['target_data']).squeeze()
ct_target_name = ct_equation_data['target_name']

# Prepare coordinate limits and mask
ct_coord_limit_params = COORD_TRANSFORM_COORD_LIMIT_CONFIG[COORD_TRANSFORM_DATASET[1]]
ct_feature_limits = ct_coord_limit_params.get('feature_limits', {})
ct_target_limits = ct_coord_limit_params.get('target_limits', {})

ct_coord_mask = np.ones(len(ct_coord_datasets['t']), dtype=bool)

# Use model-coordinate limits so both scatter views show the same selected points
for feature_name in COORD_TRANSFORM_MASK_FEATURE_KEYS:
    if feature_name in ct_feature_limits:
        feature_data = np.asarray(ct_feature_df[feature_name])
        feature_min, feature_max = ct_feature_limits[feature_name]
        ct_coord_mask &= (feature_data >= feature_min) & (feature_data <= feature_max)

if ct_target_name in ct_target_limits:
    target_min, target_max = ct_target_limits[ct_target_name]
    ct_coord_mask &= (ct_target_data >= target_min) & (ct_target_data <= target_max)

# Prepare selected interval
ct_time_mask = np.zeros(len(ct_coord_datasets['t']), dtype=bool)

for interval_idx in COORD_TRANSFORM_INTERVAL_IDXS:
    ct_window_start, ct_window_end = ct_time_intervals[interval_idx]
    ct_time_mask |= (
        (ct_coord_datasets['t'] >= ct_window_start)
        & (ct_coord_datasets['t'] <= ct_window_end)
    )

ct_data_mask = ct_time_mask & ct_coord_mask

ct_z_data = ct_target_data[ct_data_mask]
ct_z_limits = ct_target_limits.get(ct_target_name, (np.nanmin(ct_z_data), np.nanmax(ct_z_data)))

In [ ]:
# Plot coordinate-transform scatter views for selected interval

# Plot z_ddot against x and y
ct_x_key, ct_y_key = COORD_TRANSFORM_CARTESIAN_KEYS
ct_x_data = np.asarray(ct_coord_datasets[ct_x_key])[ct_data_mask]
ct_y_data = np.asarray(ct_coord_datasets[ct_y_key])[ct_data_mask]
ct_x_limits, ct_y_limits = shared_range_limits(
    ct_x_data,
    ct_y_data,
    margin=COORD_TRANSFORM_RANGE_MARGIN,
)

fig, ax = plt.subplots(subplot_kw={'projection': '3d'})

plot_spheres(
    np.column_stack([ct_x_data, ct_y_data, ct_z_data]),
    radius=COORD_TRANSFORM_POINT_RADIUS,
    sphere_props=COORD_TRANSFORM_SCATTER_CONFIG,
    ax=ax,
)

format_axis(
    ax,
    **COORD_TRANSFORM_CARTESIAN_AXIS_CONFIG,
    xlabel=CARTESIAN_COORD_LABELS[ct_x_key],
    ylabel=CARTESIAN_COORD_LABELS[ct_y_key],
    zlabel=TARGET_LABELS.get(ct_target_name, ct_target_name),
    xlims=ct_x_limits,
    ylims=ct_y_limits,
    zlims=ct_z_limits,
)
fig.tight_layout()
fig.savefig(COORD_TRANSFORM_XY_FIG_PATH, bbox_inches='tight', pad_inches=0.3)

if CLOSE_FIGS:
    plt.close(fig)

print(f'Saved: {COORD_TRANSFORM_XY_FIG_PATH.relative_to(PROJECT_ROOT)}')

# Plot z_ddot against r and cos(theta)

ct_x_key, ct_y_key = COORD_TRANSFORM_FEATURE_KEYS
ct_x_data = np.asarray(ct_feature_df[ct_x_key])[ct_data_mask]
ct_y_data = np.asarray(ct_feature_df[ct_y_key])[ct_data_mask]

ct_x_limits = ct_feature_limits.get(ct_x_key, (np.nanmin(ct_x_data), np.nanmax(ct_x_data)))
ct_y_limits = ct_feature_limits.get(ct_y_key, (np.nanmin(ct_y_data), np.nanmax(ct_y_data)))

fig, ax = plt.subplots(subplot_kw={'projection': '3d'})

plot_spheres(
    np.column_stack([ct_x_data, ct_y_data, ct_z_data]),
    radius=COORD_TRANSFORM_POINT_RADIUS,
    sphere_props=COORD_TRANSFORM_SCATTER_CONFIG,
    ax=ax,
)

format_axis(
    ax,
    **COORD_TRANSFORM_FEATURE_AXIS_CONFIG,
    xlabel=FEATURE_LABELS.get(ct_x_key, ct_x_key),
    ylabel=FEATURE_LABELS.get(ct_y_key, ct_y_key),
    zlabel=TARGET_LABELS.get(ct_target_name, ct_target_name),
    xlims=ct_x_limits,
    ylims=ct_y_limits,
    zlims=ct_z_limits,
)
fig.tight_layout()
fig.savefig(COORD_TRANSFORM_FEATURE_FIG_PATH, bbox_inches='tight', pad_inches=0.3)

if CLOSE_FIGS:
    plt.close(fig)

print(f'Saved: {COORD_TRANSFORM_FEATURE_FIG_PATH.relative_to(PROJECT_ROOT)}')